# Task 3: Python Network Intrusion Detection System (NIDS)
This notebook demonstrates how to construct a lightweight Network Intrusion Detection System (NIDS) in Python using the `scapy` library.

### What is an NIDS?
A Network Intrusion Detection System monitors network traffic for malicious activity or policy violations. Unlike a sniffer that only collects and displays packets, an IDS evaluates captured data against signatures (rules) or thresholds to detect anomalies, log alerts, and trigger defensive responses.

## Step 1: Loading Signature Rules
We load threshold parameters and payload keywords from a configuration dictionary (mimicking a JSON configuration). This permits us to customize detection metrics (e.g. how many packets constitute a scan).

In [ ]:
import time
from collections import defaultdict
from scapy.all import IP, TCP, UDP, ICMP, Raw

# Predefined rules dictionary
rules = {
    "syn_scan": {
        "enabled": True,
        "max_syn_packets": 5,  # Alert if more than 5 SYN packets in the window
        "window_seconds": 5,
        "description": "Potential TCP SYN Port Scan"
    },
    "icmp_flood": {
        "enabled": True,
        "max_packets": 5,
        "window_seconds": 5,
        "description": "Potential ICMP Ping Flood"
    },
    "plaintext_credentials": {
        "enabled": True,
        "keywords": ["password", "passwd", "username", "login"],
        "description": "Plaintext Credentials Transit"
    }
}

## Step 2: Setting up Tracking and Response Mechanisms
We need structures to keep track of incoming packets over time, and a way to simulate a firewall blocking response if an IP is detected violating rules.

In [ ]:
# Track packet timestamps: IP -> list of timestamps
syn_tracker = defaultdict(list)
icmp_tracker = defaultdict(list)

# Firewall blacklist simulation
blacklisted_ips = set()

def trigger_alert(alert_key, src_ip, details):
    timestamp = time.strftime('%H:%M:%S')
    desc = rules[alert_key]["description"]
    print(f"\033[91m[ALERT][{timestamp}] {desc} | Source: {src_ip} | {details}\033[0m")
    
    # Active Response
    if src_ip not in blacklisted_ips and src_ip != "127.0.0.1":
        blacklisted_ips.add(src_ip)
        print(f"\033[95m[FIREWALL] BLOCKED IP: {src_ip} has been blacklisted!\033[0m")

## Step 3: Designing the Packet Analyzer
This function analyzes each packet, updates the rate-limiting trackers, checks for matching patterns, and cleans old history data outside the time window.

In [ ]:
def clean_tracker(tracker, window):
    now = time.time()
    for ip in list(tracker.keys()):
        tracker[ip] = [t for t in tracker[ip] if now - t <= window]
        if not tracker[ip]:
            del tracker[ip]

def packet_analyzer(packet):
    if not packet.haslayer(IP):
        return
        
    src_ip = packet[IP].src
    dst_ip = packet[IP].dst
    now = time.time()

    # Drop packets if blacklisted
    if src_ip in blacklisted_ips:
        return

    # 1. Detect SYN Port Scan
    if rules["syn_scan"]["enabled"] and packet.haslayer(TCP):
        if packet[TCP].flags == 0x02: # Only SYN flag set
            syn_tracker[src_ip].append(now)
            window = rules["syn_scan"]["window_seconds"]
            clean_tracker(syn_tracker, window)
            
            if len(syn_tracker[src_ip]) > rules["syn_scan"]["max_syn_packets"]:
                trigger_alert(
                    "syn_scan", 
                    src_ip, 
                    f"Sent {len(syn_tracker[src_ip])} SYN packets in last {window}s."
                )
                syn_tracker[src_ip] = [] # Reset tracker

    # 2. Detect ICMP Ping Flood
    if rules["icmp_flood"]["enabled"] and packet.haslayer(ICMP):
        if packet[ICMP].type == 8: # Echo Request
            icmp_tracker[src_ip].append(now)
            window = rules["icmp_flood"]["window_seconds"]
            clean_tracker(icmp_tracker, window)
            
            if len(icmp_tracker[src_ip]) > rules["icmp_flood"]["max_packets"]:
                trigger_alert(
                    "icmp_flood", 
                    src_ip, 
                    f"Sent {len(icmp_tracker[src_ip])} ping requests in last {window}s."
                )
                icmp_tracker[src_ip] = [] # Reset tracker

    # 3. Detect Plaintext Credentials
    if rules["plaintext_credentials"]["enabled"] and packet.haslayer(Raw):
        payload = packet[Raw].load.decode('utf-8', errors='ignore')
        if any(keyword in payload.lower() for keyword in rules["plaintext_credentials"]["keywords"]):
            if "login" in payload.lower() or "password" in payload.lower():
                trigger_alert(
                    "plaintext_credentials",
                    src_ip,
                    f"Payload match to {dst_ip}: {payload[:80].strip()}"
                )

## Step 4: Simulating Traffic to Test Alerts
Let's write a testing script using Scapy's object constructor to generate simulated packets locally and pass them to our analyzer. This demonstrates how the engine processes and catches threats without requiring live external attacks.

In [ ]:
from scapy.all import Ether

print("[*] Simulation Starting...\n")

# Test 1: Testing Plaintext Credentials
print("--- Test Scenario A: Sending Plaintext credentials ---")
post_data = "POST /login HTTP/1.1\r\nHost: example.com\r\nContent-Length: 35\r\n\r\nusername=admin&password=mySuperPassword123"
credential_packet = IP(src="192.168.1.50", dst="192.168.1.1")/TCP()/Raw(load=post_data)
packet_analyzer(credential_packet)

# Test 2: Testing SYN Port Scan Simulation
print("\n--- Test Scenario B: Simulating a Port Scan ---")
for port in range(80, 87):
    # Send SYN packets (flags=S) to different ports from IP 192.168.1.60
    scan_packet = IP(src="192.168.1.60", dst="192.168.1.1")/TCP(dport=port, flags="S")
    packet_analyzer(scan_packet)
    time.sleep(0.1)

# Test 3: Verify Blacklisted IP (Further packets from 192.168.1.60 should be dropped)
print("\n--- Test Scenario C: Testing Firewall Block ---")
blocked_packet = IP(src="192.168.1.60", dst="192.168.1.1")/TCP(dport=443, flags="S")
# This shouldn't print an alert because the source IP is now in the blacklisted_ips set
packet_analyzer(blocked_packet)
print("[Info] Packets from 192.168.1.60 evaluated. If no alert printed, block was successful!")